# Clasificación de imágenes — MNIST y Los Simpson  
### Aumentación sintética, PyTorch y Keras

**Objetivo didáctico.** Recorrer un flujo completo de clasificación de imágenes usando dos datasets muy distintos, comparando dos frameworks (PyTorch y TensorFlow/Keras) y explorando el efecto de:

- Diferentes **arquitecturas** (MLP, CNN superficial, CNN profunda con `BatchNorm` + `Dropout`).
- Diferentes **optimizadores** (`SGD`, `Adam`, `RMSprop`).
- Diferentes **learning rates** y tamaños de las capas.
- **Aumentación sintética** (rotaciones, traslaciones, ruido, elastic deformation, cutout).

### Índice
1. Configuración e imports.
2. **Parte 1 — MNIST**  
   2.1 Carga · 2.2 EDA · 2.3 Aumentación sintética · 2.4 PyTorch · 2.5 Keras · 2.6 Métricas y curvas.
3. **Parte 2 — Los Simpson**  
   3.1 Carga · 3.2 EDA · 3.3 Aumentación · 3.4 PyTorch · 3.5 Keras · 3.6 Métricas.
4. Conclusiones.


## 0. Instalación e imports


In [ ]:
# Descomentar si es necesario
# !pip install torch torchvision tensorflow scikit-learn matplotlib seaborn scipy pillow tqdm


In [ ]:
import os, math, time, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from scipy.ndimage import gaussian_filter, map_coordinates

# ---------- PyTorch ----------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torchvision
from torchvision import transforms

# ---------- Keras / TensorFlow ----------
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------- Métricas ----------
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); tf.random.set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}  |  device: {DEVICE}')
print(f'TensorFlow: {tf.__version__}  |  GPU: {len(tf.config.list_physical_devices("GPU"))>0}')


---

# 🅐 Parte 1 — MNIST


## 1.1 Carga del dataset


In [ ]:
# torchvision descarga MNIST y lo guarda en ./data
mnist_train = torchvision.datasets.MNIST(root='./data', train=True,  download=True)
mnist_test  = torchvision.datasets.MNIST(root='./data', train=False, download=True)

X_train_full = mnist_train.data.numpy().astype(np.float32) / 255.0   # (60000, 28, 28)
y_train_full = mnist_train.targets.numpy()
X_test       = mnist_test.data.numpy().astype(np.float32) / 255.0    # (10000, 28, 28)
y_test       = mnist_test.targets.numpy()

# Reservamos un pedazo para validación
val_idx = np.random.RandomState(SEED).choice(len(X_train_full), 6000, replace=False)
mask = np.ones(len(X_train_full), dtype=bool); mask[val_idx] = False
X_train, y_train = X_train_full[mask], y_train_full[mask]
X_val,   y_val   = X_train_full[val_idx], y_train_full[val_idx]

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'Clases: {np.unique(y_train)}')


## 1.2 EDA — Análisis exploratorio


In [ ]:
# Distribución de clases
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, y, name in zip(axes, [y_train, y_val, y_test], ['Train', 'Val', 'Test']):
    counts = pd.Series(y).value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=sns.color_palette('viridis', 10))
    ax.set_title(f'{name} ({len(y):,} muestras)')
    ax.set_xlabel('dígito'); ax.set_ylabel('conteo')
plt.tight_layout(); plt.show()


In [ ]:
# Ejemplos por clase
fig, axes = plt.subplots(10, 8, figsize=(10, 12))
for cls in range(10):
    idxs = np.where(y_train == cls)[0][:8]
    for j, idx in enumerate(idxs):
        axes[cls, j].imshow(X_train[idx], cmap='gray')
        axes[cls, j].axis('off')
        if j == 0:
            axes[cls, j].set_ylabel(f'{cls}', rotation=0, labelpad=15, fontsize=13, va='center')
plt.suptitle('Ejemplos por clase', y=1.0, fontsize=14)
plt.tight_layout(); plt.show()


In [ ]:
# Imagen promedio y desviación estándar por clase → variabilidad intra-clase
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
for cls in range(10):
    imgs = X_train[y_train == cls]
    axes[0, cls].imshow(imgs.mean(axis=0), cmap='gray')
    axes[0, cls].set_title(f'{cls}')
    axes[0, cls].axis('off')
    axes[1, cls].imshow(imgs.std(axis=0), cmap='magma')
    axes[1, cls].axis('off')
axes[0, 0].set_ylabel('media',  rotation=0, labelpad=25, fontsize=10, va='center')
axes[1, 0].set_ylabel('desv.',  rotation=0, labelpad=25, fontsize=10, va='center')
plt.suptitle('Imagen promedio (fila 1) y desviación estándar (fila 2) por clase')
plt.tight_layout(); plt.show()


In [ ]:
# Estadísticas globales
stats = pd.DataFrame({
    'split':   ['train', 'val', 'test'],
    'n':       [len(X_train), len(X_val), len(X_test)],
    'pix_min': [X_train.min(), X_val.min(), X_test.min()],
    'pix_max': [X_train.max(), X_val.max(), X_test.max()],
    'pix_mean':[X_train.mean(), X_val.mean(), X_test.mean()],
    'pix_std': [X_train.std(),  X_val.std(),  X_test.std()],
})
stats


## 1.3 Aumentación sintética

Implementamos manualmente las transformaciones más comunes para entender qué hace cada una. Todas trabajan sobre un `np.ndarray` de forma `(H, W)`.

| Transformación | Idea |
|---------------|------|
| `rotate`      | rotación aleatoria en ±θ° |
| `translate`   | desplazamiento en píxeles |
| `zoom`        | zoom in/out con crop-resize |
| `add_noise`   | ruido gaussiano |
| `elastic_deform` | deformación elástica (Simard et al. 2003) |
| `cutout`      | recorte cuadrado a cero |
| `random_augment` | pipeline aleatorio que combina las anteriores |


In [ ]:
from scipy.ndimage import rotate as nd_rotate, shift as nd_shift, zoom as nd_zoom

def aug_rotate(img, max_angle=25):
    angle = np.random.uniform(-max_angle, max_angle)
    return nd_rotate(img, angle, reshape=False, mode='constant', cval=0.0)

def aug_translate(img, max_pix=4):
    dx, dy = np.random.uniform(-max_pix, max_pix, 2)
    return nd_shift(img, shift=(dy, dx), mode='constant', cval=0.0)

def aug_zoom(img, zoom_range=(0.85, 1.15)):
    z = np.random.uniform(*zoom_range)
    zoomed = nd_zoom(img, z, mode='constant', cval=0.0)
    h, w = img.shape
    zh, zw = zoomed.shape
    out = np.zeros_like(img)
    if z >= 1:  # crop centrado
        y0 = (zh - h) // 2; x0 = (zw - w) // 2
        out = zoomed[y0:y0+h, x0:x0+w]
    else:       # pad centrado
        y0 = (h - zh) // 2; x0 = (w - zw) // 2
        out[y0:y0+zh, x0:x0+zw] = zoomed
    return out

def aug_noise(img, sigma=0.08):
    return np.clip(img + np.random.normal(0, sigma, img.shape), 0, 1)

def aug_elastic(img, alpha=34, sigma=4):
    """Deformación elástica al estilo Simard 2003."""
    shape = img.shape
    dx = gaussian_filter(np.random.uniform(-1, 1, shape), sigma) * alpha
    dy = gaussian_filter(np.random.uniform(-1, 1, shape), sigma) * alpha
    y, x = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), indexing='ij')
    coords = np.stack([np.clip(y + dy, 0, shape[0]-1),
                       np.clip(x + dx, 0, shape[1]-1)])
    return map_coordinates(img, coords, order=1, mode='reflect')

def aug_cutout(img, size=8):
    h, w = img.shape
    cy = np.random.randint(0, h); cx = np.random.randint(0, w)
    y0, y1 = max(0, cy - size//2), min(h, cy + size//2)
    x0, x1 = max(0, cx - size//2), min(w, cx + size//2)
    out = img.copy(); out[y0:y1, x0:x1] = 0
    return out

def random_augment(img):
    """Pipeline compuesto: aplica cada transformación con cierta probabilidad."""
    if np.random.rand() < 0.7: img = aug_rotate(img, 20)
    if np.random.rand() < 0.5: img = aug_translate(img, 3)
    if np.random.rand() < 0.4: img = aug_zoom(img)
    if np.random.rand() < 0.3: img = aug_elastic(img)
    if np.random.rand() < 0.3: img = aug_noise(img, 0.05)
    if np.random.rand() < 0.2: img = aug_cutout(img, 6)
    return img.astype(np.float32)


In [ ]:
# Visualizamos cada transformación por separado
sample = X_train[7]  # un dígito de ejemplo
transforms_demo = [
    ('Original',         sample),
    ('Rotate ±25°',      aug_rotate(sample, 25)),
    ('Translate ±4px',   aug_translate(sample, 4)),
    ('Zoom 0.85-1.15',   aug_zoom(sample)),
    ('Gaussian noise',   aug_noise(sample, 0.1)),
    ('Elastic deform',   aug_elastic(sample)),
    ('Cutout',           aug_cutout(sample, 8)),
    ('Composición',      random_augment(sample)),
]
fig, axes = plt.subplots(1, len(transforms_demo), figsize=(16, 2.2))
for ax, (name, im) in zip(axes, transforms_demo):
    ax.imshow(im, cmap='gray'); ax.set_title(name, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# Generamos varias versiones aumentadas de un mismo dígito
fig, axes = plt.subplots(4, 10, figsize=(14, 6))
base_idxs = np.random.RandomState(0).choice(len(X_train), 4, replace=False)
for i, idx in enumerate(base_idxs):
    axes[i, 0].imshow(X_train[idx], cmap='gray')
    axes[i, 0].set_title(f'orig ({y_train[idx]})', fontsize=9)
    axes[i, 0].axis('off')
    for j in range(1, 10):
        axes[i, j].imshow(random_augment(X_train[idx]), cmap='gray')
        axes[i, j].set_title(f'aug {j}', fontsize=8)
        axes[i, j].axis('off')
plt.suptitle('Datos sintéticos generados con random_augment')
plt.tight_layout(); plt.show()


In [ ]:
# Preparamos un train set aumentado ampliado (2× el original)
N_AUG = len(X_train)
aug_idx = np.random.choice(len(X_train), N_AUG, replace=True)
X_aug = np.stack([random_augment(X_train[i]) for i in tqdm(aug_idx, desc='augmenting')])
y_aug = y_train[aug_idx]

X_train_ext = np.concatenate([X_train, X_aug], axis=0)
y_train_ext = np.concatenate([y_train, y_aug], axis=0)
perm = np.random.permutation(len(X_train_ext))
X_train_ext = X_train_ext[perm]; y_train_ext = y_train_ext[perm]
print(f'Train aumentado: {X_train_ext.shape}')


## 1.4 Clasificación con PyTorch

Definimos tres arquitecturas de dificultad creciente y las entrenamos con distintos optimizadores y learning rates.


In [ ]:
class MLP_MNIST(nn.Module):
    def __init__(self, hidden=(256, 128), n_classes=10):
        super().__init__()
        h1, h2 = hidden
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, h1), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(h1, h2),    nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(h2, n_classes),
        )
    def forward(self, x): return self.net(x)


class ShallowCNN_MNIST(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14x14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*7*7, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))


class DeepCNN_MNIST(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),                              # 14x14

            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),                              # 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, n_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))


In [ ]:
def make_torch_loader(X, y, batch=128, shuffle=True):
    Xt = torch.tensor(X, dtype=torch.float32).unsqueeze(1)  # (N,1,28,28)
    yt = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch, shuffle=shuffle, num_workers=0)

def train_torch(model, tr_loader, va_loader, epochs, optim_name='adam', lr=1e-3):
    model = model.to(DEVICE)
    if optim_name == 'adam':    opt = torch.optim.Adam(model.parameters(), lr=lr)
    elif optim_name == 'sgd':   opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif optim_name == 'rmsprop': opt = torch.optim.RMSprop(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    for ep in range(epochs):
        model.train(); tl, tc, tn = 0, 0, 0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            out = model(xb); loss = crit(out, yb)
            loss.backward(); opt.step()
            tl += loss.item()*xb.size(0); tc += (out.argmax(1)==yb).sum().item(); tn += xb.size(0)
        model.eval(); vl, vc, vn = 0, 0, 0
        with torch.no_grad():
            for xb, yb in va_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb); vl += crit(out, yb).item()*xb.size(0)
                vc += (out.argmax(1)==yb).sum().item(); vn += xb.size(0)
        hist['train_loss'].append(tl/tn); hist['val_loss'].append(vl/vn)
        hist['train_acc'].append(tc/tn);  hist['val_acc'].append(vc/vn)
    return model, hist


def eval_torch(model, loader):
    model.eval(); ys, ps, prs = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs  = F.softmax(logits, dim=1).cpu().numpy()
            ys.append(yb.numpy()); ps.append(logits.argmax(1).cpu().numpy()); prs.append(probs)
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(prs)


In [ ]:
# Usamos un subset del train aumentado para acelerar la búsqueda de hiperparámetros
SUB = 20000
sub_idx = np.random.choice(len(X_train_ext), SUB, replace=False)
Xs, ys = X_train_ext[sub_idx], y_train_ext[sub_idx]

tr_loader_pt = make_torch_loader(Xs, ys, batch=128)
va_loader_pt = make_torch_loader(X_val, y_val, batch=256, shuffle=False)
te_loader_pt = make_torch_loader(X_test, y_test, batch=256, shuffle=False)

# Experimentos: (arquitectura, optimizador, learning rate)
experiments = [
    ('MLP',        'adam',    1e-3),
    ('MLP',        'sgd',     1e-2),
    ('ShallowCNN', 'adam',    1e-3),
    ('ShallowCNN', 'rmsprop', 5e-4),
    ('DeepCNN',    'adam',    1e-3),
    ('DeepCNN',    'sgd',     1e-2),
]

torch_results = {}
EPOCHS = 6  # Ajustar según recursos

for arch, opt_name, lr in experiments:
    tag = f'{arch}_{opt_name}_lr{lr}'
    print(f'\n▶ {tag}')
    net = {'MLP': MLP_MNIST, 'ShallowCNN': ShallowCNN_MNIST, 'DeepCNN': DeepCNN_MNIST}[arch]()
    t0 = time.time()
    net, hist = train_torch(net, tr_loader_pt, va_loader_pt, epochs=EPOCHS, optim_name=opt_name, lr=lr)
    dur = time.time() - t0
    yt, pt, prt = eval_torch(net, te_loader_pt)
    torch_results[tag] = {
        'arch': arch, 'optim': opt_name, 'lr': lr,
        'hist': hist,
        'test_acc':  accuracy_score(yt, pt),
        'test_prec': precision_score(yt, pt, average='macro'),
        'test_rec':  recall_score(yt, pt, average='macro'),
        'test_f1':   f1_score(yt, pt, average='macro'),
        'y_true': yt, 'y_pred': pt, 'y_prob': prt,
        'n_params': sum(p.numel() for p in net.parameters()),
        'time_sec': dur, 'model': net,
    }
    print(f'  acc={torch_results[tag]["test_acc"]:.4f} | f1={torch_results[tag]["test_f1"]:.4f} | {dur:.1f}s')


In [ ]:
torch_summary = pd.DataFrame([
    {'run': k, 'arch': v['arch'], 'optim': v['optim'], 'lr': v['lr'],
     'test_acc': v['test_acc'], 'test_prec': v['test_prec'],
     'test_rec': v['test_rec'], 'test_f1': v['test_f1'],
     'n_params': v['n_params'], 'time_sec': round(v['time_sec'], 1)}
    for k, v in torch_results.items()
]).sort_values('test_acc', ascending=False)
torch_summary.reset_index(drop=True)


## 1.5 Clasificación con Keras / TensorFlow

Definimos las mismas tres arquitecturas en Keras y probamos otras combinaciones de optimizadores y learning rates.


In [ ]:
def build_mlp_keras(hidden=(256, 128)):
    m = keras.Sequential([
        layers.Input((28, 28, 1)), layers.Flatten(),
        layers.Dense(hidden[0], activation='relu'), layers.Dropout(0.2),
        layers.Dense(hidden[1], activation='relu'), layers.Dropout(0.2),
        layers.Dense(10, activation='softmax'),
    ], name='MLP_keras')
    return m

def build_shallow_cnn_keras():
    m = keras.Sequential([
        layers.Input((28, 28, 1)),
        layers.Conv2D(16, 3, padding='same', activation='relu'), layers.MaxPool2D(),
        layers.Conv2D(32, 3, padding='same', activation='relu'), layers.MaxPool2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'), layers.Dropout(0.3),
        layers.Dense(10, activation='softmax'),
    ], name='ShallowCNN_keras')
    return m

def build_deep_cnn_keras():
    m = keras.Sequential([
        layers.Input((28, 28, 1)),
        layers.Conv2D(32, 3, padding='same'), layers.BatchNormalization(), layers.ReLU(),
        layers.Conv2D(32, 3, padding='same'), layers.BatchNormalization(), layers.ReLU(),
        layers.MaxPool2D(), layers.Dropout(0.25),
        layers.Conv2D(64, 3, padding='same'), layers.BatchNormalization(), layers.ReLU(),
        layers.Conv2D(64, 3, padding='same'), layers.BatchNormalization(), layers.ReLU(),
        layers.MaxPool2D(), layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(256), layers.BatchNormalization(), layers.ReLU(), layers.Dropout(0.4),
        layers.Dense(10, activation='softmax'),
    ], name='DeepCNN_keras')
    return m

def get_keras_optim(name, lr):
    if name == 'adam':    return keras.optimizers.Adam(lr)
    if name == 'sgd':     return keras.optimizers.SGD(lr, momentum=0.9)
    if name == 'rmsprop': return keras.optimizers.RMSprop(lr)
    raise ValueError(name)


In [ ]:
# Preparamos datos con canal (para Keras/Conv2D)
Xs_k  = Xs[..., None]
Xv_k  = X_val[..., None]
Xt_k  = X_test[..., None]

keras_experiments = [
    ('MLP',        'adam',    1e-3),
    ('MLP',        'rmsprop', 1e-3),
    ('ShallowCNN', 'adam',    1e-3),
    ('ShallowCNN', 'sgd',     1e-2),
    ('DeepCNN',    'adam',    5e-4),
    ('DeepCNN',    'rmsprop', 1e-3),
]

builders = {
    'MLP': build_mlp_keras,
    'ShallowCNN': build_shallow_cnn_keras,
    'DeepCNN': build_deep_cnn_keras,
}

keras_results = {}
for arch, opt_name, lr in keras_experiments:
    tag = f'{arch}_{opt_name}_lr{lr}'
    print(f'\n▶ {tag}')
    tf.keras.backend.clear_session()
    model = builders[arch]()
    model.compile(optimizer=get_keras_optim(opt_name, lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    t0 = time.time()
    hist = model.fit(Xs_k, ys, validation_data=(Xv_k, y_val),
                     epochs=EPOCHS, batch_size=128, verbose=0)
    dur = time.time() - t0

    y_prob = model.predict(Xt_k, verbose=0)
    y_pred = y_prob.argmax(1)

    keras_results[tag] = {
        'arch': arch, 'optim': opt_name, 'lr': lr,
        'hist': hist.history,
        'test_acc':  accuracy_score(y_test, y_pred),
        'test_prec': precision_score(y_test, y_pred, average='macro'),
        'test_rec':  recall_score(y_test, y_pred, average='macro'),
        'test_f1':   f1_score(y_test, y_pred, average='macro'),
        'y_true': y_test, 'y_pred': y_pred, 'y_prob': y_prob,
        'n_params': model.count_params(),
        'time_sec': dur, 'model': model,
    }
    print(f'  acc={keras_results[tag]["test_acc"]:.4f} | f1={keras_results[tag]["test_f1"]:.4f} | {dur:.1f}s')


In [ ]:
keras_summary = pd.DataFrame([
    {'run': k, 'arch': v['arch'], 'optim': v['optim'], 'lr': v['lr'],
     'test_acc': v['test_acc'], 'test_prec': v['test_prec'],
     'test_rec': v['test_rec'], 'test_f1': v['test_f1'],
     'n_params': v['n_params'], 'time_sec': round(v['time_sec'], 1)}
    for k, v in keras_results.items()
]).sort_values('test_acc', ascending=False)
keras_summary.reset_index(drop=True)


## 1.6 Métricas y curvas de aprendizaje


In [ ]:
# Comparación cruzada entre los mejores modelos
combined = []
for src, results in [('torch', torch_results), ('keras', keras_results)]:
    for k, v in results.items():
        combined.append({'framework': src, 'run': k, **{m: v[m] for m in ['test_acc','test_prec','test_rec','test_f1']}})
comparison = pd.DataFrame(combined).sort_values('test_acc', ascending=False).reset_index(drop=True)
comparison.head(12)


In [ ]:
# Curvas de aprendizaje — PyTorch
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for tag, res in torch_results.items():
    axes[0].plot(res['hist']['val_loss'], label=tag)
    axes[1].plot(res['hist']['val_acc'],  label=tag)
axes[0].set_title('PyTorch — Loss de validación')
axes[1].set_title('PyTorch — Accuracy de validación')
for a in axes: a.set_xlabel('epoch'); a.legend(fontsize=8)
axes[0].set_ylabel('cross-entropy'); axes[1].set_ylabel('accuracy')
plt.tight_layout(); plt.show()


In [ ]:
# Curvas de aprendizaje — Keras
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for tag, res in keras_results.items():
    axes[0].plot(res['hist']['val_loss'],     label=tag)
    axes[1].plot(res['hist']['val_accuracy'], label=tag)
axes[0].set_title('Keras — Loss de validación')
axes[1].set_title('Keras — Accuracy de validación')
for a in axes: a.set_xlabel('epoch'); a.legend(fontsize=8)
axes[0].set_ylabel('cross-entropy'); axes[1].set_ylabel('accuracy')
plt.tight_layout(); plt.show()


In [ ]:
# Elegimos el mejor modelo de cada framework y hacemos análisis en detalle
best_torch_key = max(torch_results, key=lambda k: torch_results[k]['test_acc'])
best_keras_key = max(keras_results, key=lambda k: keras_results[k]['test_acc'])
print(f'Mejor PyTorch: {best_torch_key}')
print(f'Mejor Keras:   {best_keras_key}')


In [ ]:
# Matriz de confusión de ambos mejores modelos
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (label, res) in zip(axes, [('PyTorch: '+best_torch_key, torch_results[best_torch_key]),
                                   ('Keras: '+best_keras_key,   keras_results[best_keras_key])]):
    cm = confusion_matrix(res['y_true'], res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=range(10), yticklabels=range(10))
    ax.set_title(label); ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
plt.tight_layout(); plt.show()


In [ ]:
# Reporte de clasificación
print('=== PyTorch ==='); print(classification_report(
    torch_results[best_torch_key]['y_true'],
    torch_results[best_torch_key]['y_pred'], digits=4))
print('=== Keras ===');   print(classification_report(
    keras_results[best_keras_key]['y_true'],
    keras_results[best_keras_key]['y_pred'], digits=4))


In [ ]:
# Curvas ROC multiclase — one-vs-rest
def plot_multiclass_roc(y_true, y_prob, n_classes, title, ax):
    y_bin = label_binarize(y_true, classes=list(range(n_classes)))
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        ax.plot(fpr, tpr, lw=1, label=f'clase {i} (AUC={auc(fpr, tpr):.3f})')

    # Curva micro (agregada)
    fpr_mi, tpr_mi, _ = roc_curve(y_bin.ravel(), y_prob.ravel())
    ax.plot(fpr_mi, tpr_mi, 'k--', lw=2, label=f'micro (AUC={auc(fpr_mi, tpr_mi):.3f})')
    ax.plot([0, 1], [0, 1], 'gray', lw=0.5)
    ax.set_title(title); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.legend(fontsize=7, loc='lower right')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
plot_multiclass_roc(torch_results[best_torch_key]['y_true'],
                    torch_results[best_torch_key]['y_prob'], 10,
                    f'ROC — {best_torch_key} (PyTorch)', axes[0])
plot_multiclass_roc(keras_results[best_keras_key]['y_true'],
                    keras_results[best_keras_key]['y_prob'], 10,
                    f'ROC — {best_keras_key} (Keras)', axes[1])
plt.tight_layout(); plt.show()


In [ ]:
# Curvas Precision-Recall multiclase (para complementar ROC)
def plot_multiclass_pr(y_true, y_prob, n_classes, title, ax):
    y_bin = label_binarize(y_true, classes=list(range(n_classes)))
    for i in range(n_classes):
        prec, rec, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
        ap = average_precision_score(y_bin[:, i], y_prob[:, i])
        ax.plot(rec, prec, lw=1, label=f'clase {i} (AP={ap:.3f})')
    ax.set_title(title); ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.legend(fontsize=7, loc='lower left')

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
plot_multiclass_pr(torch_results[best_torch_key]['y_true'],
                   torch_results[best_torch_key]['y_prob'], 10,
                   f'PR — {best_torch_key} (PyTorch)', axes[0])
plot_multiclass_pr(keras_results[best_keras_key]['y_true'],
                   keras_results[best_keras_key]['y_prob'], 10,
                   f'PR — {best_keras_key} (Keras)', axes[1])
plt.tight_layout(); plt.show()


In [ ]:
# Errores más frecuentes: pares (real, predicho) donde falla el mejor modelo
res = keras_results[best_keras_key]
errors = np.where(res['y_true'] != res['y_pred'])[0]
print(f'Errores totales: {len(errors)} de {len(res["y_true"])} ({100*len(errors)/len(res["y_true"]):.2f}%)')

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
sample_err = np.random.choice(errors, 16, replace=False)
for ax, idx in zip(axes.flat, sample_err):
    ax.imshow(X_test[idx], cmap='gray')
    ax.set_title(f'{res["y_true"][idx]}→{res["y_pred"][idx]}', fontsize=9,
                 color='red')
    ax.axis('off')
plt.suptitle(f'Ejemplos de errores del mejor modelo Keras ({best_keras_key})')
plt.tight_layout(); plt.show()


---

# 🅑 Parte 2 — Los Simpson

Repetimos el flujo con un dataset de imágenes de personajes de Los Simpson.  

**Fuente sugerida:** [Kaggle · The Simpsons Characters Dataset](https://www.kaggle.com/datasets/alexattia/the-simpsons-characters-dataset).  
Estructura esperada:
```
simpsons/
  simpsons_dataset/
    homer_simpson/     imagen1.jpg …
    bart_simpson/      imagen1.jpg …
    marge_simpson/     …
    …
```

Si el dataset no está disponible localmente, el código detecta la ausencia y **genera un conjunto sintético "caricaturesco"** con formas y colores para que las celdas de entrenamiento sigan siendo ejecutables y demostrar todo el flujo.


In [ ]:
from google.colab import files
import os

# 1. Configura tus credenciales directamente
os.environ['KAGGLE_USERNAME'] = "feliperamirezherrera"
os.environ['KAGGLE_KEY'] = "KGAT_b6015af26deac435e88f0e65512b0364"

# 3. Descarga el dataset de Kaggle
# NOTA: Por defecto estoy usando el dataset de imágenes de los personajes.
!kaggle datasets download -d alexattia/the-simpsons-characters-dataset

# 4. Descomprime el dataset de forma silenciosa (-q) en una carpeta llamada "simpsons_data"
!unzip -q the-simpsons-characters-dataset.zip -d simpsons_data
print("¡Descarga y descompresión completadas!")

## 2.1 Carga del dataset


In [ ]:
from pathlib import Path

POSSIBLE_ROOTS = [
    Path('./simpsons_data/simpsons_dataset/simpsons_dataset'), # Actual folder containing character images
    Path('./simpsons_data/kaggle_simpson_testset'), # Another potential folder for character images
    Path('./simpsons_data/simpsons_dataset'),
    Path('./simpsons_data'),
    Path('./data/simpsons_dataset'),
    Path('/kaggle/input/the-simpsons-characters-dataset/simpsons_dataset/simpsons_dataset'),
    Path.home() / 'datasets' / 'simpsons_dataset',
]

IMG_SIZE = 64            # Redimensionamos todas las imágenes a 64x64
MAX_PER_CLASS = 300      # Limitamos por clase para tiempos razonables
TOP_CLASSES = 8          # Nos quedamos con los N personajes más frecuentes

def find_simpsons_root():
    # Iterate through possible roots and return the first one that is a directory and contains subdirectories (character folders)
    for p in POSSIBLE_ROOTS:
        if p.is_dir() and any(sd.is_dir() for sd in p.iterdir()):
            return p
    return None

simpsons_root = find_simpsons_root()
print(f'Dataset detectado en: {simpsons_root}' if simpsons_root else '⚠  Dataset NO encontrado. Se generará uno sintético.')

In [ ]:
import random
from tqdm.auto import tqdm

def load_simpsons_real(root, img_size=IMG_SIZE, max_per_class=MAX_PER_CLASS, top_k=TOP_CLASSES):
    classes = sorted([d for d in root.iterdir() if d.is_dir()])
    counts = {c.name: len(list(c.glob('*.jpg')) + list(c.glob('*.png'))) for c in classes}
    top = sorted(counts.items(), key=lambda kv: -kv[1])[:top_k]
    selected = [name for name, _ in top]
    print('Clases seleccionadas:')
    for n, c in top: print(f'  {n:30s} imgs≈{c}')

    X, y, labels = [], [], selected
    for label_idx, name in enumerate(selected):
        files = list((root/name).glob('*.jpg')) + list((root/name).glob('*.png'))
        random.shuffle(files)
        files = files[:max_per_class]
        for f in tqdm(files, desc=name, leave=False):
            try:
                img = Image.open(f).convert('RGB').resize((img_size, img_size))
                X.append(np.array(img, dtype=np.float32) / 255.0)
                y.append(label_idx)
            except Exception:
                pass
    return np.array(X), np.array(y), labels


def make_synthetic_simpsons(n_classes=6, n_per_class=250, img_size=IMG_SIZE):
    """Genera un dataset caricaturesco: cada 'clase' es una cara circular con un color de piel
    característico + rasgos aleatorios (ojos, pelo). Sirve solo como fallback didáctico."""
    palette = [
        ('yellow_fam',  (250, 220, 60)),   # tonos Simpson
        ('orange_fam',  (230, 140, 60)),
        ('pink_fam',    (250, 180, 200)),
        ('green_fam',   (140, 210, 130)),
        ('blue_fam',    (140, 180, 240)),
        ('purple_fam',  (200, 150, 240)),
    ][:n_classes]
    labels = [name for name, _ in palette]

    X, y = [], []
    for cls, (_, base_rgb) in enumerate(palette):
        for _ in range(n_per_class):
            img = np.full((img_size, img_size, 3), 30, dtype=np.uint8)
            # 'cara' circular
            cy, cx = np.random.randint(20, 44, 2)
            r = np.random.randint(16, 24)
            yy, xx = np.ogrid[:img_size, :img_size]
            face = (yy - cy)**2 + (xx - cx)**2 <= r**2
            noise = np.random.randint(-25, 25, 3)
            color = np.clip(np.array(base_rgb) + noise, 0, 255).astype(np.uint8)
            img[face] = color

            # 'ojos'
            for ex, ey in [(-r//3, -r//4), (r//3, -r//4)]:
                oy, ox = cy + ey, cx + ex
                eye = (yy - oy)**2 + (xx - ox)**2 <= 3**2
                img[eye] = (255, 255, 255)
                pupil = (yy - oy)**2 + (xx - ox)**2 <= 1
                img[pupil] = (0, 0, 0)

            # 'boca'
            my = cy + r//2
            img[my:my+2, cx-r//3:cx+r//3] = (60, 20, 20)

            X.append(img.astype(np.float32) / 255.0)
            y.append(cls)
    return np.array(X), np.array(y), labels


if simpsons_root is not None:
    X_s_all, y_s_all, class_names = load_simpsons_real(simpsons_root)
else:
    X_s_all, y_s_all, class_names = make_synthetic_simpsons()

print(f'\nShape: {X_s_all.shape}, clases: {len(class_names)}')

In [ ]:
# Split train/val/test estratificado sencillo
from collections import defaultdict
def stratified_split(y, ratios=(0.7, 0.15, 0.15), seed=SEED):
    rng = np.random.RandomState(seed)
    tr, va, te = [], [], []
    for c in np.unique(y):
        idx = np.where(y == c)[0]; rng.shuffle(idx)
        n1 = int(ratios[0]*len(idx)); n2 = n1 + int(ratios[1]*len(idx))
        tr.extend(idx[:n1]); va.extend(idx[n1:n2]); te.extend(idx[n2:])
    return np.array(tr), np.array(va), np.array(te)

tr_i, va_i, te_i = stratified_split(y_s_all)
X_s_tr, y_s_tr = X_s_all[tr_i], y_s_all[tr_i]
X_s_va, y_s_va = X_s_all[va_i], y_s_all[va_i]
X_s_te, y_s_te = X_s_all[te_i], y_s_all[te_i]
N_CLS = len(class_names)
print(f'Train {X_s_tr.shape} | Val {X_s_va.shape} | Test {X_s_te.shape}')


## 2.2 EDA


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
counts = pd.Series(y_s_tr).value_counts().sort_index()
ax.bar([class_names[c] for c in counts.index], counts.values, color=sns.color_palette('tab10', N_CLS))
ax.set_title('Distribución de clases (train)')
plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()


In [ ]:
# Ejemplos por clase
n_show = 6
fig, axes = plt.subplots(N_CLS, n_show, figsize=(n_show*1.5, N_CLS*1.5))
if N_CLS == 1: axes = axes[None, :]
for cls in range(N_CLS):
    idxs = np.where(y_s_tr == cls)[0][:n_show]
    for j, idx in enumerate(idxs):
        axes[cls, j].imshow(X_s_tr[idx])
        axes[cls, j].axis('off')
        if j == 0:
            axes[cls, j].set_ylabel(class_names[cls], rotation=0, labelpad=45, va='center', fontsize=9)
plt.suptitle('Ejemplos por personaje', y=1.0)
plt.tight_layout(); plt.show()


In [ ]:
# Estadísticas de canal
means = X_s_tr.mean(axis=(0, 1, 2))
stds  = X_s_tr.std(axis=(0, 1, 2))
print('Media por canal RGB:', means.round(4))
print('Desv. por canal RGB:', stds.round(4))


## 2.3 Aumentación (imágenes RGB)


In [ ]:
def aug_rgb_rotate(img, max_angle=20):
    angle = np.random.uniform(-max_angle, max_angle)
    return nd_rotate(img, angle, axes=(0, 1), reshape=False, mode='reflect')

def aug_rgb_flip(img):
    return img[:, ::-1, :].copy() if np.random.rand() < 0.5 else img

def aug_rgb_translate(img, max_pix=6):
    dx, dy = np.random.uniform(-max_pix, max_pix, 2)
    return nd_shift(img, shift=(dy, dx, 0), mode='reflect')

def aug_rgb_brightness(img, delta=0.15):
    return np.clip(img + np.random.uniform(-delta, delta), 0, 1)

def aug_rgb_noise(img, sigma=0.03):
    return np.clip(img + np.random.normal(0, sigma, img.shape), 0, 1)

def aug_rgb_cutout(img, size=12):
    h, w, _ = img.shape
    cy = np.random.randint(0, h); cx = np.random.randint(0, w)
    y0, y1 = max(0, cy - size//2), min(h, cy + size//2)
    x0, x1 = max(0, cx - size//2), min(w, cx + size//2)
    out = img.copy(); out[y0:y1, x0:x1, :] = 0.5
    return out

def rgb_random_augment(img):
    img = aug_rgb_flip(img)
    if np.random.rand() < 0.7: img = aug_rgb_rotate(img)
    if np.random.rand() < 0.5: img = aug_rgb_translate(img)
    if np.random.rand() < 0.5: img = aug_rgb_brightness(img)
    if np.random.rand() < 0.3: img = aug_rgb_noise(img)
    if np.random.rand() < 0.3: img = aug_rgb_cutout(img)
    return img.astype(np.float32)


In [ ]:
# Visualizar transformaciones individuales
sample = X_s_tr[np.random.RandomState(1).choice(len(X_s_tr))]
demo = [
    ('Original',     sample),
    ('Flip H',       aug_rgb_flip(sample.copy())),
    ('Rotate',       aug_rgb_rotate(sample)),
    ('Translate',    aug_rgb_translate(sample)),
    ('Brightness',   aug_rgb_brightness(sample, 0.25)),
    ('Noise',        aug_rgb_noise(sample, 0.05)),
    ('Cutout',       aug_rgb_cutout(sample)),
    ('Composición',  rgb_random_augment(sample)),
]
fig, axes = plt.subplots(1, len(demo), figsize=(16, 2.2))
for ax, (n, im) in zip(axes, demo):
    ax.imshow(np.clip(im, 0, 1)); ax.set_title(n, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# Varias versiones aumentadas de un mismo ejemplo por clase
fig, axes = plt.subplots(N_CLS, 8, figsize=(12, 1.5*N_CLS))
if N_CLS == 1: axes = axes[None, :]
for cls in range(N_CLS):
    idx = np.where(y_s_tr == cls)[0][0]
    axes[cls, 0].imshow(X_s_tr[idx])
    axes[cls, 0].set_ylabel(class_names[cls], rotation=0, labelpad=40, va='center', fontsize=9)
    axes[cls, 0].set_title('orig', fontsize=8); axes[cls, 0].axis('off')
    for j in range(1, 8):
        axes[cls, j].imshow(np.clip(rgb_random_augment(X_s_tr[idx]), 0, 1))
        axes[cls, j].set_title(f'aug {j}', fontsize=8); axes[cls, j].axis('off')
plt.suptitle('Imágenes sintéticas generadas', y=1.0)
plt.tight_layout(); plt.show()


In [ ]:
# Ampliamos el train aumentando cada imagen 2 veces
aug_X, aug_y = [], []
for _ in range(2):
    for i in range(len(X_s_tr)):
        aug_X.append(rgb_random_augment(X_s_tr[i]))
        aug_y.append(y_s_tr[i])
X_s_tr_ext = np.concatenate([X_s_tr, np.stack(aug_X)], axis=0)
y_s_tr_ext = np.concatenate([y_s_tr, np.array(aug_y)], axis=0)
perm = np.random.permutation(len(X_s_tr_ext))
X_s_tr_ext = X_s_tr_ext[perm]; y_s_tr_ext = y_s_tr_ext[perm]
print(f'Train aumentado: {X_s_tr_ext.shape}')


## 2.4 Clasificación con PyTorch


In [ ]:
class CNN_S_Small(nn.Module):
    def __init__(self, n_classes, img_size=IMG_SIZE):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        with torch.no_grad():
            flat = self.features(torch.zeros(1, 3, img_size, img_size)).numel()
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(flat, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))


class CNN_S_Deep(nn.Module):
    def __init__(self, n_classes, img_size=IMG_SIZE):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                nn.MaxPool2d(2), nn.Dropout(0.25))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128))
        with torch.no_grad():
            flat = self.features(torch.zeros(1, 3, img_size, img_size)).numel()
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(flat, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, n_classes))
    def forward(self, x): return self.classifier(self.features(x))


In [ ]:
def make_rgb_loader(X, y, batch=64, shuffle=True):
    Xt = torch.tensor(X.transpose(0, 3, 1, 2), dtype=torch.float32)  # (N,C,H,W)
    yt = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch, shuffle=shuffle)

tr_s_pt = make_rgb_loader(X_s_tr_ext, y_s_tr_ext, batch=64)
va_s_pt = make_rgb_loader(X_s_va,     y_s_va,     batch=128, shuffle=False)
te_s_pt = make_rgb_loader(X_s_te,     y_s_te,     batch=128, shuffle=False)

simpson_torch_exps = [
    ('CNN_Small', 'adam',   1e-3),
    ('CNN_Small', 'sgd',    1e-2),
    ('CNN_Deep',  'adam',   1e-3),
    ('CNN_Deep',  'rmsprop',5e-4),
]
EPOCHS_S = 8

simpson_torch_results = {}
for arch, opt_name, lr in simpson_torch_exps:
    tag = f'{arch}_{opt_name}_lr{lr}'
    print(f'\n▶ {tag}')
    net = {'CNN_Small': CNN_S_Small, 'CNN_Deep': CNN_S_Deep}[arch](n_classes=N_CLS)
    t0 = time.time()
    net, hist = train_torch(net, tr_s_pt, va_s_pt, epochs=EPOCHS_S, optim_name=opt_name, lr=lr)
    dur = time.time() - t0
    yt, pt, prt = eval_torch(net, te_s_pt)
    simpson_torch_results[tag] = {
        'arch': arch, 'optim': opt_name, 'lr': lr, 'hist': hist,
        'test_acc': accuracy_score(yt, pt),
        'test_prec': precision_score(yt, pt, average='macro', zero_division=0),
        'test_rec':  recall_score(yt, pt, average='macro', zero_division=0),
        'test_f1':   f1_score(yt, pt, average='macro', zero_division=0),
        'y_true': yt, 'y_pred': pt, 'y_prob': prt,
        'n_params': sum(p.numel() for p in net.parameters()),
        'time_sec': dur, 'model': net,
    }
    print(f'  acc={simpson_torch_results[tag]["test_acc"]:.4f} | f1={simpson_torch_results[tag]["test_f1"]:.4f} | {dur:.1f}s')

pd.DataFrame([
    {'run': k, **{m: v[m] for m in ['arch','optim','lr','test_acc','test_prec','test_rec','test_f1','n_params']}}
    for k, v in simpson_torch_results.items()
]).sort_values('test_acc', ascending=False).reset_index(drop=True)


## 2.5 Clasificación con Keras


In [ ]:
def build_simpson_small_keras(n_classes, img_size=IMG_SIZE):
    return keras.Sequential([
        layers.Input((img_size, img_size, 3)),
        layers.Conv2D(32, 3, padding='same', activation='relu'), layers.MaxPool2D(),
        layers.Conv2D(64, 3, padding='same', activation='relu'), layers.MaxPool2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'), layers.Dropout(0.3),
        layers.Dense(n_classes, activation='softmax'),
    ], name='CNN_S_Small_keras')

def build_simpson_deep_keras(n_classes, img_size=IMG_SIZE):
    def block(x, c):
        x = layers.Conv2D(c, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
        x = layers.Conv2D(c, 3, padding='same')(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
        return layers.Dropout(0.25)(layers.MaxPool2D()(x))
    inp = layers.Input((img_size, img_size, 3))
    x = block(inp, 32); x = block(x, 64); x = block(x, 128)
    x = layers.Flatten()(x)
    x = layers.Dense(256)(x); x = layers.BatchNormalization()(x); x = layers.ReLU()(x); x = layers.Dropout(0.4)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    return keras.Model(inp, out, name='CNN_S_Deep_keras')

simpson_keras_exps = [
    ('CNN_Small', 'adam',    1e-3),
    ('CNN_Small', 'rmsprop', 1e-3),
    ('CNN_Deep',  'adam',    1e-3),
    ('CNN_Deep',  'sgd',     1e-2),
]
simpson_builders_k = {'CNN_Small': build_simpson_small_keras, 'CNN_Deep': build_simpson_deep_keras}

simpson_keras_results = {}
for arch, opt_name, lr in simpson_keras_exps:
    tag = f'{arch}_{opt_name}_lr{lr}'
    print(f'\n▶ {tag}')
    tf.keras.backend.clear_session()
    m = simpson_builders_k[arch](N_CLS)
    m.compile(optimizer=get_keras_optim(opt_name, lr),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
    t0 = time.time()
    hist = m.fit(X_s_tr_ext, y_s_tr_ext, validation_data=(X_s_va, y_s_va),
                 epochs=EPOCHS_S, batch_size=64, verbose=0)
    dur = time.time() - t0
    y_prob = m.predict(X_s_te, verbose=0)
    y_pred = y_prob.argmax(1)
    simpson_keras_results[tag] = {
        'arch': arch, 'optim': opt_name, 'lr': lr, 'hist': hist.history,
        'test_acc': accuracy_score(y_s_te, y_pred),
        'test_prec': precision_score(y_s_te, y_pred, average='macro', zero_division=0),
        'test_rec':  recall_score(y_s_te, y_pred, average='macro', zero_division=0),
        'test_f1':   f1_score(y_s_te, y_pred, average='macro', zero_division=0),
        'y_true': y_s_te, 'y_pred': y_pred, 'y_prob': y_prob,
        'n_params': m.count_params(), 'time_sec': dur, 'model': m,
    }
    print(f'  acc={simpson_keras_results[tag]["test_acc"]:.4f} | f1={simpson_keras_results[tag]["test_f1"]:.4f} | {dur:.1f}s')

pd.DataFrame([
    {'run': k, **{m: v[m] for m in ['arch','optim','lr','test_acc','test_prec','test_rec','test_f1','n_params']}}
    for k, v in simpson_keras_results.items()
]).sort_values('test_acc', ascending=False).reset_index(drop=True)


## 2.6 Métricas y visualizaciones


In [ ]:
# Curvas de aprendizaje — Simpsons
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for tag, res in simpson_torch_results.items():
    axes[0, 0].plot(res['hist']['val_loss'], label=tag)
    axes[0, 1].plot(res['hist']['val_acc'],  label=tag)
for tag, res in simpson_keras_results.items():
    axes[1, 0].plot(res['hist']['val_loss'],     label=tag)
    axes[1, 1].plot(res['hist']['val_accuracy'], label=tag)
axes[0, 0].set_title('PyTorch — Loss val');  axes[0, 1].set_title('PyTorch — Acc val')
axes[1, 0].set_title('Keras   — Loss val');  axes[1, 1].set_title('Keras   — Acc val')
for a in axes.flat: a.set_xlabel('epoch'); a.legend(fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
best_s_torch = max(simpson_torch_results, key=lambda k: simpson_torch_results[k]['test_acc'])
best_s_keras = max(simpson_keras_results, key=lambda k: simpson_keras_results[k]['test_acc'])
print(f'Mejor PyTorch: {best_s_torch}')
print(f'Mejor Keras:   {best_s_keras}')


In [ ]:
# Matrices de confusión
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (label, res) in zip(axes, [('PyTorch: '+best_s_torch, simpson_torch_results[best_s_torch]),
                                   ('Keras: '+best_s_keras,   simpson_keras_results[best_s_keras])]):
    cm = confusion_matrix(res['y_true'], res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False,
                xticklabels=class_names, yticklabels=class_names)
    ax.set_title(label); ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()


In [ ]:
# Reportes
print('=== PyTorch ==='); print(classification_report(
    simpson_torch_results[best_s_torch]['y_true'],
    simpson_torch_results[best_s_torch]['y_pred'],
    target_names=class_names, digits=4, zero_division=0))
print('=== Keras ===');   print(classification_report(
    simpson_keras_results[best_s_keras]['y_true'],
    simpson_keras_results[best_s_keras]['y_pred'],
    target_names=class_names, digits=4, zero_division=0))


In [ ]:
# ROC multiclase para Simpsons
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
plot_multiclass_roc(simpson_torch_results[best_s_torch]['y_true'],
                    simpson_torch_results[best_s_torch]['y_prob'], N_CLS,
                    f'ROC — {best_s_torch} (PyTorch)', axes[0])
plot_multiclass_roc(simpson_keras_results[best_s_keras]['y_true'],
                    simpson_keras_results[best_s_keras]['y_prob'], N_CLS,
                    f'ROC — {best_s_keras} (Keras)', axes[1])
plt.tight_layout(); plt.show()


In [ ]:
# Ejemplos de predicciones (correctas e incorrectas)
res = simpson_keras_results[best_s_keras]
correct = np.where(res['y_true'] == res['y_pred'])[0]
wrong   = np.where(res['y_true'] != res['y_pred'])[0]

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
sample_c = np.random.choice(correct, min(8, len(correct)), replace=False)
sample_w = np.random.choice(wrong,   min(8, len(wrong)),   replace=False) if len(wrong) else []

for j, idx in enumerate(sample_c):
    axes[0, j].imshow(X_s_te[idx])
    axes[0, j].set_title(f'{class_names[res["y_true"][idx]]}', fontsize=8, color='green')
    axes[0, j].axis('off')
for j, idx in enumerate(sample_w):
    axes[1, j].imshow(X_s_te[idx])
    axes[1, j].set_title(f'{class_names[res["y_true"][idx]]}→{class_names[res["y_pred"][idx]]}',
                          fontsize=7, color='red')
    axes[1, j].axis('off')
for j in range(len(sample_w), 8):
    axes[1, j].axis('off')

axes[0, 0].set_ylabel('OK',    rotation=0, labelpad=20, fontsize=10, color='green')
axes[1, 0].set_ylabel('Error', rotation=0, labelpad=20, fontsize=10, color='red')
plt.suptitle(f'Predicciones — {best_s_keras}')
plt.tight_layout(); plt.show()


---

## 3. Conclusiones

**Sobre la aumentación sintética**
- Rotaciones, traslaciones y deformaciones elásticas ayudan a generalizar cuando los datos son limitados (visible sobre todo en el dataset de Simpsons con pocas imágenes por clase).
- Cutout y ruido gaussiano actúan como regularización implícita: fuerzan al modelo a no depender de píxeles específicos.

**PyTorch vs Keras**
- Ambos frameworks alcanzan métricas equivalentes con las mismas arquitecturas.
- Keras destaca por su brevedad al declarar arquitecturas secuenciales; PyTorch da más control sobre el bucle de entrenamiento (útil para debugging y experimentación avanzada).

**Optimizadores y learning rate**
- `Adam` con `lr=1e-3` es un default sólido para arrancar.
- `SGD + momentum` con `lr` mayor (`1e-2`) suele igualar o superar a Adam cuando se dispone de más epochs y `lr scheduling`.
- `RMSprop` es competitivo en redes convolucionales, especialmente con `lr` intermedios (`5e-4` – `1e-3`).

**Arquitecturas**
- MLP es un buen baseline para MNIST pero **no escala** a Simpsons (donde la información espacial es fundamental).
- Las CNN profundas con `BatchNorm` y `Dropout` son las que mejor generalizan, a costa de mayor tiempo de entrenamiento.

**Métricas relevantes**
- Accuracy es informativo cuando las clases están balanceadas (MNIST).
- Precision, Recall y F1 macro son imprescindibles cuando hay desbalance (Simpsons).
- ROC y PR multiclase (one-vs-rest) ayudan a diagnosticar clases problemáticas: una curva plana revela una clase que el modelo confunde sistemáticamente.

**Sugerencias de extensión**
1. Añadir `EarlyStopping` y `ReduceLROnPlateau` a Keras y `torch.optim.lr_scheduler` a PyTorch.
2. Usar `torchvision.transforms` o `tf.keras.layers.RandomFlip / RandomRotation` para aumentación en GPU.
3. Probar transfer learning en Simpsons con un backbone preentrenado (`ResNet18`, `MobileNetV2`).
